In [ ]:
# Instalação das dependências
%pip install -q numpy pandas scikit-learn matplotlib seaborn tqdm


In [ ]:
# Imports necessários
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuração de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Adiciona o diretório scripts ao path
sys.path.append('scripts')

print("✅ Ambiente configurado com sucesso!")


In [ ]:
# Importa o módulo de experimento
from run_experiment import FiloTransformerExperiment

# Cria instância do experimento
experiment = FiloTransformerExperiment()

# Carrega os dados
texts, labels = experiment.load_data()

# Estatísticas básicas
print("📊 ESTATÍSTICAS DO DATASET PHEME")
print("="*40)
print(f"Total de amostras: {len(texts):,}")
print(f"Rumores (Fake News): {sum(labels):,} ({sum(labels)/len(labels)*100:.1f}%)")
print(f"Não-rumores: {len(labels) - sum(labels):,} ({(len(labels) - sum(labels))/len(labels)*100:.1f}%)")
print(f"Tamanho médio do texto: {np.mean([len(text.split()) for text in texts]):.1f} palavras")

# Visualização da distribuição
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
categories = ['Não-rumor', 'Rumor']
counts = [len(labels) - sum(labels), sum(labels)]
colors = ['#2ecc71', '#e74c3c']
ax1.bar(categories, counts, color=colors, alpha=0.8)
ax1.set_title('Distribuição de Classes', fontsize=14, fontweight='bold')
ax1.set_ylabel('Número de Amostras')
for i, v in enumerate(counts):
    ax1.text(i, v + 50, str(v), ha='center', fontweight='bold')

# Gráfico de pizza
ax2.pie(counts, labels=categories, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Proporção de Classes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Mostra exemplos de cada classe
print("🔍 EXEMPLOS DE TEXTOS\n")

# Exemplos de não-rumores
print("✅ NÃO-RUMORES (Textos Verdadeiros):")
print("-" * 50)
non_rumor_indices = [i for i, label in enumerate(labels) if label == 0]
for i in np.random.choice(non_rumor_indices, 3, replace=False):
    print(f"• {texts[i][:150]}...")
    print()

print("\n❌ RUMORES (Fake News):")
print("-" * 50)
rumor_indices = [i for i, label in enumerate(labels) if label == 1]
for i in np.random.choice(rumor_indices, 3, replace=False):
    print(f"• {texts[i][:150]}...")
    print()


In [ ]:
# Extração de características semânticas
print("🔤 Extraindo características semânticas (TF-IDF)...")
semantic_features = experiment.extract_semantic_features(texts)
print(f"✅ Dimensão das características semânticas: {semantic_features.shape}")
print(f"   • Número de amostras: {semantic_features.shape[0]:,}")
print(f"   • Número de características: {semantic_features.shape[1]:,}")


In [ ]:
# Extração de características filogenéticas
print("\n🌳 Extraindo características filogenéticas (TAGs)...")
phylogenetic_features = experiment.extract_phylogenetic_features(texts)
print(f"✅ Dimensão das características filogenéticas: {phylogenetic_features.shape}")
print(f"   • Número de amostras: {phylogenetic_features.shape[0]:,}")
print(f"   • Número de características: {phylogenetic_features.shape[1]:,}")

# Visualização das características filogenéticas
feature_names = [
    'Padrões de Casualidade', 'Triggers Imediatos', 'Pré-condições',
    'Apelos à Ação', 'Marcadores Temporais', 'Padrões de Localização',
    'Padrões de Persona', 'Amplificação', 'Emoção', 'Incerteza', 
    'Autoridade', 'Urgência', 'Polarização', 'Manipulação'
]

# Médias por classe
rumor_mask = np.array(labels) == 1
non_rumor_mask = ~rumor_mask

rumor_means = phylogenetic_features[rumor_mask].mean(axis=0)
non_rumor_means = phylogenetic_features[non_rumor_mask].mean(axis=0)

# Gráfico comparativo
plt.figure(figsize=(12, 6))
x = np.arange(len(feature_names))
width = 0.35

plt.bar(x - width/2, non_rumor_means, width, label='Não-rumor', color='#2ecc71', alpha=0.8)
plt.bar(x + width/2, rumor_means, width, label='Rumor', color='#e74c3c', alpha=0.8)

plt.xlabel('Características Filogenéticas')
plt.ylabel('Valor Médio')
plt.title('Comparação de Características Filogenéticas por Classe', fontsize=14, fontweight='bold')
plt.xticks(x, feature_names, rotation=45, ha='right')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Análise da diferença entre classes
differences = rumor_means - non_rumor_means
sorted_indices = np.argsort(np.abs(differences))[::-1]

print("🎯 CARACTERÍSTICAS MAIS DISCRIMINATIVAS:")
print("="*50)
for i in sorted_indices[:5]:
    diff_percent = (differences[i] / non_rumor_means[i]) * 100 if non_rumor_means[i] != 0 else 0
    print(f"{feature_names[i]:25} → {diff_percent:+6.1f}% em rumores")

# Heatmap de correlação
plt.figure(figsize=(10, 8))
correlation_matrix = np.corrcoef(phylogenetic_features.T)
sns.heatmap(correlation_matrix, 
            xticklabels=feature_names, 
            yticklabels=feature_names,
            cmap='coolwarm', 
            center=0,
            annot=True,
            fmt='.2f',
            square=True)
plt.title('Correlação entre Características Filogenéticas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Execução do experimento principal
print("🚀 INICIANDO EXPERIMENTO DE VALIDAÇÃO CRUZADA")
print("="*60)
print("• Dataset: PHEME")
print(f"• Total de amostras: {len(texts):,}")
print("• Método: 5-fold stratified cross-validation")
print("• Modelos: Baseline (TF-IDF) vs Filo-Transformer (TF-IDF + Filogenia)")
print("="*60)

# Executa o experimento
results = experiment.run_experiment()

# Exibe resultados detalhados
print("\n📊 RESULTADOS FINAIS")
print("="*60)

# Filo-Transformer
print("\n🧬 FILO-TRANSFORMER (COM CARACTERÍSTICAS FILOGENÉTICAS)")
for metric in ['ACCURACY', 'AUC', 'F1', 'RECALL']:
    values = results['filo'][metric.lower()]
    mean = np.mean(values)
    std = np.std(values)
    print(f"{metric:10}: {mean:.4f} ± {std:.4f}")

# Baseline
print("\n📊 BASELINE (APENAS CARACTERÍSTICAS SEMÂNTICAS)")
for metric in ['ACCURACY', 'AUC', 'F1', 'RECALL']:
    values = results['baseline'][metric.lower()]
    mean = np.mean(values)
    std = np.std(values)
    print(f"{metric:10}: {mean:.4f} ± {std:.4f}")

# Melhorias
print("\n🎯 MELHORIA DO FILO-TRANSFORMER")
for metric in ['ACCURACY', 'AUC', 'F1', 'RECALL']:
    filo_mean = np.mean(results['filo'][metric.lower()])
    baseline_mean = np.mean(results['baseline'][metric.lower()])
    improvement = filo_mean - baseline_mean
    improvement_pct = (improvement / baseline_mean) * 100
    print(f"{metric:10}: +{improvement:.4f} (+{improvement_pct:.1f}%)")


In [ ]:
# Comparação visual dos modelos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['accuracy', 'auc', 'f1', 'recall']
metric_names = ['Acurácia', 'AUC', 'F1-Score', 'Recall']
colors = {'baseline': '#3498db', 'filo': '#e74c3c'}

for idx, (ax, metric, name) in enumerate(zip(axes.flat, metrics, metric_names)):
    # Dados
    baseline_vals = results['baseline'][metric]
    filo_vals = results['filo'][metric]
    
    # Box plot
    bp = ax.boxplot([baseline_vals, filo_vals], 
                    labels=['Baseline', 'Filo-Transformer'],
                    patch_artist=True,
                    widths=0.6)
    
    # Cores
    bp['boxes'][0].set_facecolor(colors['baseline'])
    bp['boxes'][1].set_facecolor(colors['filo'])
    
    # Adiciona valores médios
    for i, (vals, pos) in enumerate([(baseline_vals, 1), (filo_vals, 2)]):
        mean_val = np.mean(vals)
        ax.plot(pos, mean_val, 'wo', markersize=10, markeredgecolor='black', markeredgewidth=2)
        ax.text(pos, mean_val + 0.01, f'{mean_val:.3f}', ha='center', fontweight='bold')
    
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_ylabel('Valor')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0.6, 1.0)

plt.suptitle('Comparação de Performance: Baseline vs Filo-Transformer', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Análise por fold
print("📊 DESEMPENHO POR FOLD")
print("="*70)
print(f"{'Fold':^6} | {'Baseline AUC':^12} | {'Filo AUC':^12} | {'Melhoria':^12} | {'%':^6}")
print("-"*70)

for i in range(5):
    baseline_auc = results['baseline']['auc'][i]
    filo_auc = results['filo']['auc'][i]
    improvement = filo_auc - baseline_auc
    improvement_pct = (improvement / baseline_auc) * 100
    
    print(f"{i+1:^6} | {baseline_auc:^12.4f} | {filo_auc:^12.4f} | {improvement:^12.4f} | {improvement_pct:^6.1f}")

# Gráfico de linhas mostrando evolução por fold
plt.figure(figsize=(10, 6))
folds = np.arange(1, 6)

for metric, name in zip(['auc', 'f1'], ['AUC', 'F1-Score']):
    plt.plot(folds, results['baseline'][metric], 'o-', label=f'Baseline {name}', linewidth=2, markersize=8)
    plt.plot(folds, results['filo'][metric], 's-', label=f'Filo-Transformer {name}', linewidth=2, markersize=8)

plt.xlabel('Fold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Performance por Fold de Validação Cruzada', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.xticks(folds)
plt.tight_layout()
plt.show()


In [ ]:
# Função para análise de um texto individual
def analyze_single_text(text, experiment):
    """Analisa um texto e retorna suas características filogenéticas."""
    print(f"📝 Texto: \"{text}\"\n")
    
    # Extrai características
    semantic_feat = experiment.extract_semantic_features([text])
    phylo_feat = experiment.extract_phylogenetic_features([text])
    
    # Mostra características filogenéticas
    print("🌳 CARACTERÍSTICAS FILOGENÉTICAS DETECTADAS:")
    print("-" * 50)
    
    feature_names = [
        'Padrões de Casualidade', 'Triggers Imediatos', 'Pré-condições',
        'Apelos à Ação', 'Marcadores Temporais', 'Padrões de Localização',
        'Padrões de Persona', 'Amplificação', 'Emoção', 'Incerteza', 
        'Autoridade', 'Urgência', 'Polarização', 'Manipulação'
    ]
    
    # Identifica características presentes
    for i, (name, value) in enumerate(zip(feature_names, phylo_feat[0])):
        if value > 0:
            print(f"✓ {name}: {value:.2f}")
    
    return phylo_feat[0]

# Exemplos de análise
print("🔍 ANÁLISE DE TEXTOS EXEMPLO\n")
print("="*60)

# Exemplo 1: Texto típico de fake news
fake_text = "URGENT: Government hiding vaccine deaths! 1000s dying but media silent! Share before they delete this!!!"
features1 = analyze_single_text(fake_text, experiment)

print("\n" + "="*60 + "\n")

# Exemplo 2: Texto factual
real_text = "Study shows COVID-19 vaccines have saved millions of lives worldwide according to peer-reviewed research."
features2 = analyze_single_text(real_text, experiment)

# Comparação visual
plt.figure(figsize=(10, 6))
x = np.arange(len(feature_names))
width = 0.35

plt.bar(x - width/2, features1, width, label='Fake News', color='#e74c3c', alpha=0.8)
plt.bar(x + width/2, features2, width, label='Notícia Real', color='#2ecc71', alpha=0.8)

plt.xlabel('Características Filogenéticas')
plt.ylabel('Valor')
plt.title('Comparação de Características: Fake News vs Notícia Real', fontsize=14, fontweight='bold')
plt.xticks(x, feature_names, rotation=45, ha='right')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
